In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.utils import resample

In [2]:
reference = pd.read_csv('../../ModellerModule/reference_files/DMS_substitutions_ProteinGym.csv')
saving = False

In [3]:
pd.unique(reference.selection_type)

array(['Growth', nan, 'Flow cytometry', 'survival assessment assay',
       'Antibiotics resistance', 'Flow Cytometry Assay',
       'Receptor activity', 'bulk RNA-sequencing',
       'cDNA display proteolysis', 'enzymatic activity', 'FACS',
       'Amp resistance', 'Amoxicillin resistance', 'complementation',
       'toxin activity', 'protein abundance', 'Activity, binding',
       'Binding', 'TOXCAT-Beta-lactamase (TbL) screen', 'thermostability',
       'Growth (antitoxin neutralization of ParE3)',
       'Yeast complementation', 'Resistance to statin inhibition',
       'Voltage', 'inhibitor resistance', 'Activity',
       'VAMP-seq, drug sensitivity', 'phage fitness',
       'quantification and selection of GFP-positive cells by flow cytometry after DNA damage induced by daunorubicin',
       'Protein stability', 'lipid phosphatase activity',
       'binding assays', 'activity', 'Fluorescence',
       'Survival (dosed with trametinib)', 'binding',
       'Auto-ubiquitination'], dt

## Activity Analysis

In [4]:
act_ref = reference[reference.selection_type.isin(['enzymatic activity','Activity, binding','Activity','lipid phosphatase activity','activity'])]

for idx in act_ref.index:
    print(act_ref.loc[idx,'DMS_id'])
    print(act_ref.loc[idx,'title'])
    print(act_ref.loc[idx,'selection_type'],act_ref.loc[idx,'coarse_selection_type'])
    print(act_ref.loc[idx,'selection_assay'])
    print(act_ref.loc[idx,'DMS_number_single_mutants'])
    print(act_ref.loc[idx,'DMS_number_multiple_mutants'])
    #print(entry.title)
    print('------------------------------')

ANCSZ_Hobbs_2022
Saturation mutagenesis of a predicted ancestral Syk-family kinase
enzymatic activity Activity
successful phosphorylation of bait peptide
4670
0
------------------------------
CP2C9_HUMAN_Amorosi_2021_activity
Massively parallel characterization of CYP2C9 variant enzyme activity and abundance
Activity, binding Binding
activity, binding (to fluorescent CYP probe)
6142
0
------------------------------
HXK4_HUMAN_Gersing_2022_activity
A comprehensive map of human glucokinase variant activity
enzymatic activity OrganismalFitness
functional complementation to reduced growth on glucose medium
8570
0
------------------------------
MTH3_HAEAE_RockahShmuel_2015
Systematic Mapping of Protein Mutational Space by Prolonged Drift Reveals the Deleterious Effects of Seemingly Neutral Mutations
Activity OrganismalFitness
Growth
1777
0
------------------------------
PTEN_HUMAN_Mighell_2018
A Saturation Mutagenesis Approach to Understanding PTEN Lipid Phosphatase Activity and Genotype-Ph

In [5]:
enzyme_act_ref = act_ref[act_ref.DMS_id.isin(['ANCSZ_Hobbs_2022','Q59976_STRSQ_Romero_2015','VKOR1_HUMAN_Chiasson_2020_activity'])]
enzyme_act_ref[['MSA_filename', 'MSA_start', 'MSA_end', 'MSA_len','DMS_binarization_cutoff']]

,MSA_filename,MSA_start,MSA_end,MSA_len,DMS_binarization_cutoff
15,ANCSZ_b0.4.a2m,1,627,627,-0.057412
142,Q59976_STRSQ_full_11-26-2021_b03.a2m,1,501,501,-1.000000
212,VKOR1_HUMAN_full_11-26-2021_b03.a2m,1,163,163,0.700000


In [6]:
ANCSZ_Hobbs_2022_df = pd.read_csv(f'ANCSZ_Hobbs_2022.csv')
Q59976_STRSQ_Romero_2015_df = pd.read_csv(f'Q59976_STRSQ_Romero_2015.csv')
VKOR1_HUMAN_Chiasson_2020_activity_df = pd.read_csv(f'VKOR1_HUMAN_Chiasson_2020_activity.csv')

In [37]:
pos_list = np.arange(352,628)
mut_list = []
selection_all = None
np.random.seed(42)
for i in range(10):
    random_pos = np.random.choice(pos_list, 7, replace=False)
    selection_df = pd.DataFrame(columns=ANCSZ_Hobbs_2022_df.columns.to_list()+['pos'])
    for idx in ANCSZ_Hobbs_2022_df.index:
        pos = int(ANCSZ_Hobbs_2022_df.loc[idx].mutant[1:-1])
        if pos in random_pos:
            selection_df.loc[idx] = ANCSZ_Hobbs_2022_df.loc[idx].to_list()+[pos]
    selection_40_df = resample(selection_df, random_state=42,n_samples=40,replace=False,stratify=selection_df.pos).drop(columns=['pos'])
    selection_40_df.to_csv(f'tmp/tmp_{i}.csv',index=False)
    
ref_ = enzyme_act_ref[enzyme_act_ref.DMS_id=='ANCSZ_Hobbs_2022'].iloc[0].copy()
selection_all = None
for i in range(10):
    selection_ = pd.read_csv(f'tmp/tmp_{i}.csv')
    selection_all = pd.concat([selection_all,selection_]).reset_index(drop=True) if selection_all is not None else selection_.copy()
selection_all.loc[400] = [ref_.target_seq[0]+'1'+ref_.target_seq[0]] + [ref_.target_seq] + [ref_.DMS_binarization_cutoff] + [1] + [1]
if saving:
    selection_all.to_csv(f'../lowN/ANCSZ_Hobbs_2022_selection.csv',index=False)

In [8]:
pos_list = np.arange(2,502)
np.random.seed(11)
for i in range(10):
    random_pos = np.random.choice(pos_list, 7, replace=False)
    selection_df = pd.DataFrame(columns=Q59976_STRSQ_Romero_2015_df.columns.to_list()+['pos'])
    for idx in Q59976_STRSQ_Romero_2015_df.index:
        pos = int(Q59976_STRSQ_Romero_2015_df.loc[idx].mutant[1:-1])
        if pos in random_pos:
            selection_df.loc[idx] = Q59976_STRSQ_Romero_2015_df.loc[idx].to_list()+[pos]
    selection_40_df = resample(selection_df, random_state=11,n_samples=40,replace=False,stratify=selection_df.pos).drop(columns=['pos'])
    selection_40_df.to_csv(f'tmp/tmp_{i}.csv',index=False)
ref_ = enzyme_act_ref[enzyme_act_ref.DMS_id=='Q59976_STRSQ_Romero_2015'].iloc[0].copy()
selection_all = None
for i in range(10):
    selection_ = pd.read_csv(f'tmp/tmp_{i}.csv')
    selection_all = pd.concat([selection_all,selection_]).reset_index(drop=True) if selection_all is not None else selection_.copy()
selection_all.loc[400] = [ref_.target_seq[0]+'1'+ref_.target_seq[0]] + [ref_.target_seq] + [ref_.DMS_binarization_cutoff] + [1]
if saving:
    selection_all.to_csv(f'../lowN/Q59976_STRSQ_Romero_2015_selection.csv',index=False)

In [9]:
pos_list = np.arange(3,164)
seed = 17
np.random.seed(seed)
for i in range(10):
    random_pos = np.random.choice(pos_list, 10, replace=False)
    selection_df = pd.DataFrame(columns=VKOR1_HUMAN_Chiasson_2020_activity_df.columns.to_list()+['pos'])
    for idx in VKOR1_HUMAN_Chiasson_2020_activity_df.index:
        pos = int(VKOR1_HUMAN_Chiasson_2020_activity_df.loc[idx].mutant[1:-1])
        if pos in random_pos:
            selection_df.loc[idx] = VKOR1_HUMAN_Chiasson_2020_activity_df.loc[idx].to_list()+[pos]
    selection_40_df = resample(selection_df, random_state=seed,n_samples=40,replace=False,stratify=selection_df.pos).drop(columns=['pos'])
    selection_40_df.to_csv(f'tmp/tmp_{i}.csv',index=False)
ref_ = enzyme_act_ref[enzyme_act_ref.DMS_id=='VKOR1_HUMAN_Chiasson_2020_activity'].iloc[0].copy()
selection_all = None
for i in range(10):
    selection_ = pd.read_csv(f'tmp/tmp_{i}.csv')
    selection_all = pd.concat([selection_all,selection_]).reset_index(drop=True) if selection_all is not None else selection_.copy()
selection_all.loc[400] = [ref_.target_seq[0]+'1'+ref_.target_seq[0]] + [ref_.target_seq] + [ref_.DMS_binarization_cutoff] + [1]
if saving:
    selection_all.to_csv(f'../lowN/VKOR1_HUMAN_Chiasson_2020_activity_selection.csv',index=False)

In [18]:
reference_validation = pd.DataFrame(columns=reference.columns)
for loc,enz in enumerate(['ANCSZ_Hobbs_2022','Q59976_STRSQ_Romero_2015','VKOR1_HUMAN_Chiasson_2020_activity']):
    ref_ = enzyme_act_ref[enzyme_act_ref.DMS_id==enz].iloc[0].copy()
    ref_['DMS_filename'] = f'{enz}_selection.csv'
    ref_['DMS_id'] = f'{enz}_selection'
    ref_['DMS_total_number_mutants'] = 400
    ref_['DMS_number_single_mutants'] = 400
    reference_validation.loc[loc] = ref_.to_list()

## Diverse Analysis

In [11]:
diverse_ref = reference[reference.selection_type.isin(['survival assessment assay','Receptor activity','thermostability','inhibitor resistance','Binding','binding','Protein stability','binding assays','Fluorescence'])]

In [13]:
diverse_sel = ['A4_HUMAN_Seuma_2022','SPIKE_SARS2_Starr_2020_binding','ADRB2_HUMAN_Jones_2020','ESTA_BACSU_Nutschel_2020','MK01_HUMAN_Brenan_2016','YAP1_HUMAN_Araya_2012','SC6A4_HUMAN_Young_2021']
diverse_sel_ref = diverse_ref[diverse_ref.DMS_id.isin(diverse_sel)]

In [35]:
for index,row in diverse_sel_ref.iterrows():
    print(row.DMS_filename[:-4])
    data_ = pd.read_csv(row.DMS_filename)
    selection_all = None
    np.random.seed(42)
    for i in range(10):
        selection_40_df = resample(data_,n_samples=40,replace=False).reset_index(drop=True)
        selection_all = pd.concat([selection_all,selection_40_df]).reset_index(drop=True) if selection_all is not None else selection_40_df.copy()
    selection_all.loc[400] = [row.target_seq[0]+'1'+row.target_seq[0]] + [row.target_seq] + [row.DMS_binarization_cutoff] + [1]
    if saving:
        selection_all.to_csv(f'../lowN/'+row.DMS_filename[:-4]+f'_selection.csv',index=False)
    
        

A4_HUMAN_Seuma_2022
ADRB2_HUMAN_Jones_2020
ESTA_BACSU_Nutschel_2020
MK01_HUMAN_Brenan_2016
SC6A4_HUMAN_Young_2021
SPIKE_SARS2_Starr_2020_binding
YAP1_HUMAN_Araya_2012


In [16]:
for index,row in diverse_sel_ref[diverse_sel_ref.DMS_id.isin(['A4_HUMAN_Seuma_2022','YAP1_HUMAN_Araya_2012'])].iterrows():
    print(row.DMS_filename[:-4])
    data_ = pd.read_csv(row.DMS_filename)
    data_ = data_[data_.mutant.str.count(':')==0]
    selection_all = None
    np.random.seed(42)
    for i in range(10):
        selection_40_df = resample(data_,n_samples=40,replace=False).reset_index(drop=True)
        selection_all = pd.concat([selection_all,selection_40_df]).reset_index(drop=True) if selection_all is not None else selection_40_df.copy()
    selection_all.loc[400] = [row.target_seq[0]+'1'+row.target_seq[0]] + [row.target_seq] + [row.DMS_binarization_cutoff] + [1]
    if saving:
        selection_all.to_csv(f'../lowN/'+row.DMS_filename[:-4]+f'_single_selection.csv',index=False)


A4_HUMAN_Seuma_2022
YAP1_HUMAN_Araya_2012


In [20]:
for i in range(diverse_sel_ref.shape[0]):
    ref_ = diverse_sel_ref.iloc[i].copy()
    DMS_id = ref_.DMS_id
    ref_['DMS_filename'] = f'{DMS_id}_selection.csv'
    ref_['DMS_id'] = f'{DMS_id}_selection'
    ref_['DMS_total_number_mutants'] = 400
    ref_['DMS_number_single_mutants'] = 40
    reference_validation.loc[i+3] = ref_.to_list()
for u,DMS_id in enumerate(['A4_HUMAN_Seuma_2022','YAP1_HUMAN_Araya_2012']):
    ref_ = diverse_sel_ref[diverse_sel_ref.DMS_id==DMS_id].iloc[0].copy()
    ref_['DMS_filename'] = f'{DMS_id}_single_selection.csv'
    ref_['DMS_id'] = f'{DMS_id}_single_selection'
    ref_['DMS_total_number_mutants'] = 400
    ref_['DMS_number_single_mutants'] = 1
    reference_validation.loc[i+u+4] = ref_.to_list()
reference_validation.drop(columns='Unnamed: 0')

,DMS_id,DMS_filename,UniProt_ID,taxon,source_organism,target_seq,seq_len,includes_multiple_mutants,DMS_total_number_mutants,DMS_number_single_mutants,...,MSA_num_significant_L,raw_DMS_filename,raw_DMS_phenotype_name,raw_DMS_directionality,raw_DMS_mutant_column,weight_file_name,pdb_file,ProteinGym_version,raw_mut_offset,coarse_selection_type
0,ANCSZ_Hobbs_2022_selection,ANCSZ_Hobbs_2022_selection.csv,ANCSZ,Eukaryote,reconstructed ancestor,MADSANHLPYFYGSITREEAEDYLKQGGMSDGLFLLRQSLNSLGGY...,627,False,400,400,...,0.173844,ANCSZ_Hobbs_2022.csv,DMS_value,1.0,mutant,ANCSZ_theta_0.2.npy,ANCSZ.pdb,1.0,NaN,Activity
1,Q59976_STRSQ_Romero_2015_selection,Q59976_STRSQ_Romero_2015_selection.csv,Q59976_STRSQ,Prokaryote,Streptomyces sp.,MVPAAQQTAMAPDAALTFPEGFLWGSATASYQIEGAAAEDGRTPSI...,501,False,400,400,...,1.923077,Q59976_STRSQ_Romero_2015.csv,enrichment,1.0,mutant,Q59976_STRSQ_theta_0.2.npy,Q59976_STRSQ.pdb,0.1,NaN,Activity
2,VKOR1_HUMAN_Chiasson_2020_activity_selection,VKOR1_HUMAN_Chiasson_2020_activity_selection.csv,VKOR1_HUMAN,Human,Homo sapiens,MGSTWGSPGWVRLALCLTGLVLSLYALHVKAARARDRDYRALCDVG...,163,False,400,400,...,0.763780,VKOR1_HUMAN_Chiasson_2020.csv,activity_score,1.0,variant,VKOR1_HUMAN_theta_0.2.npy,VKOR1_HUMAN.pdb,0.1,NaN,Activity
3,A4_HUMAN_Seuma_2022_selection,A4_HUMAN_Seuma_2022_selection.csv,A4_HUMAN,Human,Homo sapiens,MLPGLALLLLAAWTARALEVPTDGNAGLLAEPQIAMFCGRLNMHMN...,770,True,400,40,...,0.000000,MS_BL_BB_indels_processed_data.tsv,nscore,1.0,mutant,A4_HUMAN_theta0.2_2023-08-07_b01.npy,A4_HUMAN.pdb,1.0,NaN,Stability
4,ADRB2_HUMAN_Jones_2020_selection,ADRB2_HUMAN_Jones_2020_selection.csv,ADRB2_HUMAN,Human,Homo sapiens,MGQPGNGSAFLLAPNGSHAPDHDVTQERDEVWVVGMGIVMSLIVLA...,413,False,400,40,...,0.795918,ADRB2_HUMAN_Jones_2020.csv,0.625,1.0,mutant_id,ADRB2_HUMAN_theta_0.2.npy,ADRB2_HUMAN.pdb,0.1,NaN,Activity
5,ESTA_BACSU_Nutschel_2020_selection,ESTA_BACSU_Nutschel_2020_selection.csv,ESTA_BACSU,Prokaryote,Bacillus subtilis,MKFVKRRIIALVTILMLSVTSLFALQPSAKAAEHNPVVMVHGIGGA...,212,False,400,40,...,1.780488,ESTA_BACSU_Nutschel_2020.csv,T50,1.0,Variants of BsLipA,ESTA_BACSU_theta_0.2.npy,ESTA_BACSU.pdb,0.1,NaN,Stability
6,MK01_HUMAN_Brenan_2016_selection,MK01_HUMAN_Brenan_2016_selection.csv,MK01_HUMAN,Human,Homo sapiens,MAAAAAAGAGPEMVRGQVFDVGPRYTNLSYIGEGAYGMVCSAYDNV...,360,False,400,40,...,0.989655,MK01_HUMAN_Brenan_2016.csv,DOX_Average,-1.0,mutant,MK01_HUMAN_theta_0.2.npy,MK01_HUMAN.pdb,0.1,NaN,OrganismalFitness
7,SC6A4_HUMAN_Young_2021_selection,SC6A4_HUMAN_Young_2021_selection.csv,SC6A4_HUMAN,Human,Homo sapiens,METTPLNSQKQLSACEDGEDCQENGVLQKVVPTPGDKVESGQISNG...,630,False,400,40,...,0.548323,SC6A4_HUMAN_Young_2021.csv,avg_MYC,1.0,mutant,SC6A4_HUMAN_theta_0.2.npy,SC6A4_HUMAN.pdb,0.1,NaN,Activity
8,SPIKE_SARS2_Starr_2020_binding_selection,SPIKE_SARS2_Starr_2020_binding_selection.csv,SPIKE_SARS2,Virus,SARS-COV2,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...,1273,False,400,40,...,1.619984,SPIKE_SARS2_Starr_2020.csv,bind_avg,1.0,mutation,SPIKE_SARS2_theta_0.01.npy,SPIKE_SARS2.pdb,0.1,NaN,Binding
9,YAP1_HUMAN_Araya_2012_selection,YAP1_HUMAN_Araya_2012_selection.csv,YAP1_HUMAN,Human,Homo sapiens,MDPGQQPPPQPAPQGQGQPPSQPPQGQGPPSGPGQPAPAATQAAPQ...,504,True,400,40,...,0.002309,YAP1_HUMAN_Araya_2012.csv,W,1.0,mutant,YAP1_HUMAN_theta_0.2.npy,YAP1_HUMAN.pdb,0.1,NaN,Binding


In [21]:
if saving:
    reference_validation.to_csv('../../ModellerModule/reference_files/DMS_substitutions_selection.csv')

# Effect of Number of Variants

In [30]:
saving_effect = True

In [41]:
for index,row in enzyme_act_ref.iterrows():
    print(row.DMS_filename[:-4])
    data_ = pd.read_csv(row.DMS_filename)
    np.random.seed(42)
    for n in [10,20,40,75,100,200]:
        selection_all = None
        for i in range(10):
            selection_40_df = resample(data_,n_samples=n,replace=False).reset_index(drop=True)
            selection_all = pd.concat([selection_all,selection_40_df]).reset_index(drop=True) if selection_all is not None else selection_40_df.copy()
        selection_all = selection_all[['mutant','mutated_sequence','DMS_score','DMS_score_bin']]
        selection_all.loc[400] = [row.target_seq[0]+'1'+row.target_seq[0]] + [row.target_seq] + [row.DMS_binarization_cutoff] + [1]
        if saving_effect:
            selection_all.to_csv(f'../lowN/'+row.DMS_filename[:-4]+f'_n{n}.csv',index=False)

ANCSZ_Hobbs_2022
Q59976_STRSQ_Romero_2015
VKOR1_HUMAN_Chiasson_2020_activity


In [51]:
reference_Neffect = pd.DataFrame(columns=reference.columns)
for loc,enz in enumerate(['ANCSZ_Hobbs_2022','Q59976_STRSQ_Romero_2015','VKOR1_HUMAN_Chiasson_2020_activity']):
    for i,n in enumerate([10,20,40,75,100,200]):
        ref_ = enzyme_act_ref[enzyme_act_ref.DMS_id==enz].iloc[0].copy()
        ref_['DMS_filename'] = f'{enz}_n{n}.csv'
        ref_['DMS_id'] = f'{enz}_n{n}'
        ref_['DMS_total_number_mutants'] = n*10
        ref_['DMS_number_single_mutants'] = n*10
        reference_Neffect.loc[(loc*6)+i] = ref_.to_list()
reference_Neffect = reference_Neffect.drop(columns='Unnamed: 0')
if saving_effect:
    reference_Neffect.to_csv('../../ModellerModule/reference_files/DMS_substitutions_Neffect.csv')